# Gemma 3 1B IT: vLLM-neuron on AWS Trainium2

This notebook deploys **google/gemma-3-1b-it** on AWS Trainium2 (trn2.3xlarge) using **vLLM-neuron**
with continuous batching, then benchmarks E2E latency and throughput matching the customer's
actual workload parameters.

### Architecture Challenges

Gemma 3 1B has non-standard architecture parameters that require workarounds on Neuron:

| Parameter | Gemma 3 1B | Standard | Impact |
|-----------|-----------|----------|--------|
| head_dim | **256** | 64/128 | NKI kernel unsupported, compiler OOB for CTE < 512 |
| vocab_size | **262144** | 32000-65536 | Required code fix in NxDI |
| sliding_window | 512 (pattern=6) | None | Interacts with k_cache_transposed |
| num_kv_heads | 1 (GQA 4:1) | 4-8 | repeat_kv layout sensitivity |

These issues are fixed in our **NxDI fork** (`fix/gemma3-1b-oob` branch). Without this fork,
Gemma 3 1B cannot run on Neuron.

### Customer Workload

| Parameter | Customer Actual | Our Config |
|-----------|----------------|------------|
| Input tokens | avg=156, max=231 | ~163 |
| Output tokens | avg=7.3, max=64 | max_tokens=64 |
| max_model_len | 512 | 512 |
| temperature | 0.0 | 0.0 |
| Quantization | INT4-AWQ | BF16 (AWQ unavailable on Neuron) |
| Speculative | ngram (3 tokens) | None (unavailable on Neuron) |

### Time to Complete

- **First run**: ~10 minutes (compilation + model load)
- **Cached run**: ~3 minutes (load only)
- **Benchmark**: ~7 minutes (6 concurrency levels x 60s each)

## Prerequisites

**Instance**: trn2.3xlarge (also works on inf2.xlarge/8xlarge with TP=1)

**AMI**: `Deep Learning AMI Neuron (Ubuntu 24.04) 20260227` (SDK 2.28)

**Disk**: 300 GB EBS (gp3)

**LNC Mode**: LNC=2 (default) -- gives 4 logical cores with 24 GB HBM each

**venv**: `/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate`

**Launch command**:
```bash
# trn2 requires a capacity block -- see project docs for capacity block purchase steps
aws ec2 run-instances \
  --image-id <ami-id> \
  --instance-type trn2.3xlarge \
  --key-name <your-key> \
  --instance-market-options '{"MarketType": "capacity-block"}' \
  --capacity-reservation-specification '{"CapacityReservationTarget": {"CapacityReservationId": "<cr-id>"}}' \
  --block-device-mappings '[{"DeviceName":"/dev/sda1","Ebs":{"VolumeSize":300,"VolumeType":"gp3"}}]' \
  --tag-specifications 'ResourceType=instance,Tags=[{Key=Name,Value=OC-Gemma-Neuron}]'
```

**HuggingFace token**: Required for Gemma 3 model access. Set `HF_TOKEN` env var.

## Step 0: Environment Detection

In [1]:
import subprocess, os, sys, json

# Check venv
VENV_PATH = '/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13'
if not os.environ.get('VIRTUAL_ENV', '').startswith(VENV_PATH):
    print(f'WARNING: Activate the vLLM venv before running this notebook:')
    print(f'  source {VENV_PATH}/bin/activate')
    print(f'  jupyter notebook')

# Detect instance type
try:
    token = subprocess.check_output(
        ['curl', '-s', '-X', 'PUT', 'http://169.254.169.254/latest/api/token',
         '-H', 'X-aws-ec2-metadata-token-ttl-seconds: 21600', '--connect-timeout', '2'],
    ).decode().strip()
    instance_type = subprocess.check_output(
        ['curl', '-s', '-H', f'X-aws-ec2-metadata-token: {token}',
         'http://169.254.169.254/latest/meta-data/instance-type', '--connect-timeout', '2'],
    ).decode().strip()
except:
    instance_type = 'unknown'
print(f'Instance: {instance_type}')

# Detect Neuron cores
neuron_ls = subprocess.check_output(['neuron-ls']).decode()
print(f'\nneuron-ls output:\n{neuron_ls}')

nc_count = neuron_ls.count('NeuronCore')
print(f'Detected {nc_count} logical NeuronCores')

if 'trn2' in instance_type:
    lnc = os.environ.get('NEURON_LOGICAL_NC_CONFIG', '2')
    print(f'LNC mode: {lnc} ({nc_count} logical cores)')

# SDK version
import torch
import torch_neuronx
print(f'\nPyTorch: {torch.__version__}')
print(f'torch-neuronx: {torch_neuronx.__version__}')

# Check HF token
hf_token = os.environ.get('HF_TOKEN', '')
if hf_token:
    print(f'HF_TOKEN: set ({len(hf_token)} chars)')
else:
    print('WARNING: HF_TOKEN not set. Required for Gemma 3 model access.')
    print('Set it with: export HF_TOKEN=hf_...')

Instance: trn2.3xlarge

neuron-ls output:
instance-type: trn2.3xlarge
instance-id: i-04e77a04e8d9212e9
logical-neuroncore-config: 2
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 4      | 0-3      | 96 GB  | 0000:33:00.0 | 0-11     | 0    |
+--------+--------+----------+--------+--------------+----------+------+

Detected 0 logical NeuronCores
LNC mode: 2 (0 logical cores)



PyTorch: 2.9.0+cu128
torch-neuronx: 2.9.0.2.12.22436+0f1dac25
HF_TOKEN: set (37 chars)


## Step 1: Install NxDI Fork (Required)

Gemma 3 1B requires fixes not yet in upstream NxDI. We install our fork's
`fix/gemma3-1b-oob` branch which contains two critical fixes:

1. **Chunked attention for head_dim>128** + vocab_size fix + auto-disable NKI kernel
2. **k_cache_transposed fix** for SWA+GQA layers (repeat_kv layout mismatch)

Without these fixes, Gemma 3 1B crashes with out-of-bounds memory access or
shape mismatch errors on Neuron.

In [2]:
%%bash
set -e

FORK_DIR="/home/ubuntu/nxdi-fork"
FORK_URL="https://github.com/jimburtoft/neuronx-distributed-inference.git"
BRANCH="fix/gemma3-1b-oob"

if [ -d "$FORK_DIR" ]; then
    echo "Fork directory exists, updating..."
    cd $FORK_DIR
    git fetch origin
    git checkout $BRANCH
    git pull origin $BRANCH
else
    echo "Cloning NxDI fork..."
    git clone -b $BRANCH $FORK_URL $FORK_DIR
fi

cd $FORK_DIR
echo "Branch: $(git branch --show-current)"
echo "Commit: $(git log -1 --oneline)"

# Install in development mode (overrides system NxDI)
pip install -e . --quiet 2>&1 | tail -3

# Verify installation
python -c "import neuronx_distributed_inference; print(f'NxDI version: {neuronx_distributed_inference.__version__}')"
echo "NxDI fork installed successfully."

Fork directory exists, updating...


Already on 'fix/gemma3-1b-oob'


Your branch is up to date with 'origin/fix/gemma3-1b-oob'.


From https://github.com/jimburtoft/neuronx-distributed-inference
 * branch            fix/gemma3-1b-

oob -> FETCH_HEAD


Already up to date.


Branch: fix/gemma3-1b-oob


Commit: 37cad61 Fix k_cache_transposed support for Gemma3 SWA+GQA layers


NxDI version: 0.8.0


NxDI fork installed successfully.


## Step 2: Download Model Locally

We download the model to a local path to avoid the NxDI trailing-slash issue
(`normalize_path()` in `application_base.py` appends `/` to HuggingFace hub IDs,
which HF rejects).

In [3]:
import os
from huggingface_hub import snapshot_download

MODEL_HF_ID = 'google/gemma-3-1b-it'
MODEL_LOCAL_PATH = '/home/ubuntu/models/gemma-3-1b-it'

if os.path.exists(os.path.join(MODEL_LOCAL_PATH, 'config.json')):
    print(f'Model already downloaded at {MODEL_LOCAL_PATH}')
else:
    print(f'Downloading {MODEL_HF_ID} to {MODEL_LOCAL_PATH}...')
    snapshot_download(
        MODEL_HF_ID,
        local_dir=MODEL_LOCAL_PATH,
        token=os.environ.get('HF_TOKEN'),
    )
    print('Download complete.')

# Verify
config_path = os.path.join(MODEL_LOCAL_PATH, 'config.json')
import json
with open(config_path) as f:
    config = json.load(f)
text_config = config.get('text_config', config)
print(f"\nModel config:")
print(f"  head_dim: {text_config.get('head_dim', 'N/A')}")
print(f"  vocab_size: {text_config.get('vocab_size', config.get('vocab_size', 'N/A'))}")
print(f"  num_attention_heads: {text_config.get('num_attention_heads', 'N/A')}")
print(f"  num_key_value_heads: {text_config.get('num_key_value_heads', 'N/A')}")
print(f"  sliding_window: {text_config.get('sliding_window', 'N/A')}")
print(f"  query_pre_attn_scalar: {text_config.get('query_pre_attn_scalar', 'N/A')}")

Model already downloaded at /home/ubuntu/models/gemma-3-1b-it

Model config:
  head_dim: 256
  vocab_size: 262144
  num_attention_heads: 4
  num_key_value_heads: 1
  sliding_window: 512
  query_pre_attn_scalar: 256


## Step 3: Configure and Launch vLLM-neuron

### Critical Configuration Notes

| Setting | Value | Why |
|---------|-------|-----|
| `attn_kernel_enabled` | **false** | NKI kernel doesn't support head_dim=256 |
| `k_cache_transposed` | **true** | Required for correct K cache layout with our SWA+GQA fix |
| `context_encoding_buckets` | **[512]** | CTE buckets < 512 cause DGE OOB with head_dim=256 (compiler issue) |
| `token_generation_buckets` | **[512]** | Same compiler constraint |
| `on_device_sampling_config` | **null** | Disable on-device sampling for compatibility |
| `batch_size` | **4** | Best latency; BS16 tested and was strictly worse for this workload |
| `n_active_tokens` | **4** | Must match batch_size |
| `seq_len` | **512** | Matches max_model_len |
| `tp_degree` | **1** | 1B model fits on single core |

In [4]:
import os, json

# --- Model and serving configuration ---
MODEL_PATH = '/home/ubuntu/models/gemma-3-1b-it'
MAX_MODEL_LEN = 512
MAX_NUM_SEQS = 4
DTYPE = 'bfloat16'
TP_DEGREE = 1
BATCH_SIZE = 4
PORT = 8000

# --- Neuron-specific override config ---
# This is the TESTED, WORKING configuration for Gemma 3 1B on Neuron.
# Do not change these settings without understanding the architectural constraints.
NEURON_OVERRIDE_CONFIG = {
    'override_neuron_config': {
        'tp_degree': TP_DEGREE,
        'batch_size': BATCH_SIZE,
        'seq_len': MAX_MODEL_LEN,
        'n_active_tokens': BATCH_SIZE,
        'context_encoding_buckets': [512],  # MUST be >= 512 (compiler issue with head_dim=256)
        'token_generation_buckets': [512],   # Same constraint
        'on_device_sampling_config': None,    # Disable on-device sampling
        'attn_kernel_enabled': False,          # NKI kernel doesn't support head_dim=256
        'k_cache_transposed': True,            # Required for our SWA+GQA fix
    }
}

print('Neuron override config:')
print(json.dumps(NEURON_OVERRIDE_CONFIG, indent=2, default=str))

print(f'\nServer will listen on port {PORT}')
print(f'Model: {MODEL_PATH}')
print(f'Batch size: {BATCH_SIZE} | max_model_len: {MAX_MODEL_LEN} | TP: {TP_DEGREE}')

Neuron override config:
{
  "override_neuron_config": {
    "tp_degree": 1,
    "batch_size": 4,
    "seq_len": 512,
    "n_active_tokens": 4,
    "context_encoding_buckets": [
      512
    ],
    "token_generation_buckets": [
      512
    ],
    "on_device_sampling_config": null,
    "attn_kernel_enabled": false,
    "k_cache_transposed": true
  }
}

Server will listen on port 8000
Model: /home/ubuntu/models/gemma-3-1b-it
Batch size: 4 | max_model_len: 512 | TP: 1


In [5]:
import subprocess, os, json

# Build the --additional-config JSON string
additional_config_str = json.dumps(NEURON_OVERRIDE_CONFIG)

# Write launch script
launch_script = f"""#!/bin/bash
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
export VLLM_NEURON_FRAMEWORK=neuronx-distributed-inference
export VLLM_RPC_TIMEOUT=300000
export HF_TOKEN={os.environ.get('HF_TOKEN', '')}
export NEURON_CC_FLAGS='--retry_failed_compilation'

python -m vllm.entrypoints.openai.api_server \\
  --model {MODEL_PATH} \\
  --tensor-parallel-size {TP_DEGREE} \\
  --max-model-len {MAX_MODEL_LEN} \\
  --max-num-seqs {MAX_NUM_SEQS} \\
  --dtype {DTYPE} \\
  --port {PORT} \\
  --disable-log-requests \\
  --no-enable-prefix-caching \\
  --additional-config '{additional_config_str}'
"""

script_path = '/home/ubuntu/launch_vllm.sh'
with open(script_path, 'w') as f:
    f.write(launch_script)
os.chmod(script_path, 0o755)
print(f'Launch script written to {script_path}')

# Kill any existing vLLM server
subprocess.run(['pkill', '-f', 'vllm.entrypoints'], capture_output=True)
import time; time.sleep(2)

# Launch in background
log_path = '/home/ubuntu/vllm_server.log'
log_file = open(log_path, 'w')
server_proc = subprocess.Popen(
    ['/bin/bash', script_path],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
print(f'Server PID: {server_proc.pid}')
print(f'Log: {log_path}')
print(f'\nCompilation will take ~5-10 min on first run, ~1-3 min from cache.')

Launch script written to /home/ubuntu/launch_vllm.sh


Server PID: 10987
Log: /home/ubuntu/vllm_server.log

Compilation will take ~5-10 min on first run, ~1-3 min from cache.


In [6]:
import time, requests

print('Waiting for vLLM-neuron to compile and start...')
start = time.time()
timeout = 1800  # 30 minutes max

while time.time() - start < timeout:
    # Check if process died
    if server_proc.poll() is not None:
        print(f'\nERROR: Server process died with code {server_proc.returncode}')
        print('Last 30 lines of log:')
        with open('/home/ubuntu/vllm_server.log') as f:
            lines = f.readlines()
            print(''.join(lines[-30:]))
        raise RuntimeError('vLLM server failed to start')
    
    try:
        resp = requests.get(f'http://localhost:{PORT}/health', timeout=2)
        if resp.status_code == 200:
            elapsed = time.time() - start
            print(f'\nvLLM-neuron is READY! ({elapsed:.0f}s = {elapsed/60:.1f} min)')
            break
    except:
        pass
    
    time.sleep(10)
    elapsed = time.time() - start
    if int(elapsed) % 60 < 10:
        with open('/home/ubuntu/vllm_server.log') as f:
            lines = f.readlines()
            last_line = lines[-1].strip() if lines else '(no output)'
        print(f'  [{elapsed/60:.0f} min] {last_line[:120]}')
else:
    print(f'TIMEOUT after {timeout/60:.0f} min')
    raise TimeoutError('vLLM-neuron failed to start')

# Verify model listing
resp = requests.get(f'http://localhost:{PORT}/v1/models')
models = resp.json()
model_id = models['data'][0]['id']
print(f'Serving model: {model_id}')

Waiting for vLLM-neuron to compile and start...


  [1 min] (APIServer pid=10988) INFO:     Application startup complete.

vLLM-neuron is READY! (60s = 1.0 min)
Serving model: /home/ubuntu/models/gemma-3-1b-it


## Step 4: Validate Model Output

In [7]:
import requests, json, time

# Get the model ID as registered by vLLM (uses local path)
resp = requests.get(f'http://localhost:{PORT}/v1/models')
MODEL_ID = resp.json()['data'][0]['id']
print(f'Model ID: {MODEL_ID}')

# Test request
url = f'http://localhost:{PORT}/v1/chat/completions'
payload = {
    'model': MODEL_ID,
    'messages': [{'role': 'user', 'content': 'What is 2+2? Answer with just the number.'}],
    'max_tokens': 10,
    'temperature': 0.0,
}

start = time.perf_counter()
resp = requests.post(url, json=payload)
elapsed = (time.perf_counter() - start) * 1000
result = resp.json()

if 'choices' in result:
    text = result['choices'][0]['message']['content']
    usage = result.get('usage', {})
    print(f'Response: "{text}"')
    print(f'Prompt tokens: {usage.get("prompt_tokens", "N/A")}')
    print(f'Completion tokens: {usage.get("completion_tokens", "N/A")}')
    print(f'Latency: {elapsed:.1f}ms')
else:
    print(f'ERROR: {json.dumps(result, indent=2)}')

Model ID: /home/ubuntu/models/gemma-3-1b-it
Response: "4
"
Prompt tokens: 22
Completion tokens: 3
Latency: 134.6ms


## Step 5: Benchmark -- Customer Workload

Matching the customer's actual workload:
- **Input**: ~155 tokens (product categorization prompt)
- **Output**: max_tokens=64, temp=0.0 (model typically generates ~2 tokens for classification)
- **Concurrency**: 1, 2, 4, 8, 16, 32
- **Duration**: 60 seconds per level

This is a **short-input, very-short-output classification/extraction workload** --
the model outputs a single category word. Performance is dominated by context encoding (CTE),
not token generation.

In [8]:
import asyncio, aiohttp, time, json, numpy as np
from datetime import datetime

VLLM_URL = f'http://localhost:{PORT}/v1/chat/completions'
MAX_TOKENS = 64
TEMPERATURE = 0.0
DURATION_SECONDS = 60
CONCURRENCY_LEVELS = [1, 2, 4, 8, 16, 32]

# Prompt: ~155 tokens (matches customer avg of 156)
# Classification/extraction task (matches short output pattern)
PROMPT = """You are a product categorizer for an e-commerce platform. Given the product description below, output only the single most appropriate category name from the standard product taxonomy.

Product: Samsung Galaxy S24 Ultra 5G smartphone with 6.8-inch Dynamic AMOLED 2X display, Snapdragon 8 Gen 3 processor, 12GB RAM, 256GB storage, 200MP main camera with OIS and advanced AI photo editing features, S Pen stylus included, titanium frame, IP68 water and dust resistance, 5000mAh battery with 45W wired and 15W wireless fast charging, running Android 14 with Samsung One UI 6.1 interface.

Category:"""

async def send_request(session, url, payload):
    start = time.perf_counter()
    try:
        async with session.post(url, json=payload, timeout=aiohttp.ClientTimeout(total=120)) as resp:
            data = await resp.json()
            elapsed_ms = (time.perf_counter() - start) * 1000
            if resp.status == 200 and 'choices' in data:
                usage = data.get('usage', {})
                return elapsed_ms, usage.get('completion_tokens', 0), usage.get('prompt_tokens', 0), data['choices'][0]['message']['content']
            return None, 0, 0, f'ERROR: {resp.status}'
    except Exception as e:
        return None, 0, 0, f'EXCEPTION: {e}'

async def worker(session, url, payload, results, stop_event):
    while not stop_event.is_set():
        latency, gen_tok, prompt_tok, text = await send_request(session, url, payload)
        if latency is not None:
            results.append((latency, gen_tok, prompt_tok, text))

async def benchmark_concurrency(num_workers):
    payload = {
        'model': MODEL_ID,
        'messages': [{'role': 'user', 'content': PROMPT}],
        'max_tokens': MAX_TOKENS,
        'temperature': TEMPERATURE,
    }
    results = []
    stop_event = asyncio.Event()

    # Warmup
    print(f'  Warmup...')
    async with aiohttp.ClientSession() as session:
        for _ in range(3):
            await send_request(session, VLLM_URL, payload)

    print(f'  Running for {DURATION_SECONDS}s with {num_workers} concurrent workers...')
    async with aiohttp.ClientSession() as session:
        workers = [
            asyncio.create_task(worker(session, VLLM_URL, payload, results, stop_event))
            for _ in range(num_workers)
        ]
        await asyncio.sleep(DURATION_SECONDS)
        stop_event.set()
        await asyncio.sleep(5)
        for w in workers:
            w.cancel()
        await asyncio.gather(*workers, return_exceptions=True)

    if not results:
        print('  No successful requests!')
        return None

    latencies = np.array([r[0] for r in results])
    gen_tokens = np.array([r[1] for r in results])
    prompt_tokens = np.array([r[2] for r in results])

    stats = {
        'concurrency': num_workers,
        'total_requests': len(results),
        'avg_prompt_tokens': round(float(prompt_tokens.mean()), 1),
        'avg_gen_tokens': round(float(gen_tokens.mean()), 1),
        'e2e_avg_ms': round(float(latencies.mean()), 1),
        'e2e_p50_ms': round(float(np.percentile(latencies, 50)), 1),
        'e2e_p90_ms': round(float(np.percentile(latencies, 90)), 1),
        'e2e_p99_ms': round(float(np.percentile(latencies, 99)), 1),
        'e2e_min_ms': round(float(latencies.min()), 1),
        'e2e_max_ms': round(float(latencies.max()), 1),
        'throughput_req_s': round(len(results) / DURATION_SECONDS, 2),
        'throughput_tok_s': round(float(gen_tokens.sum()) / DURATION_SECONDS, 1),
    }

    if num_workers == CONCURRENCY_LEVELS[0]:
        print(f'  Sample output: "{results[0][3][:80]}"')

    print(f'  Requests: {stats["total_requests"]} | Prompt: ~{stats["avg_prompt_tokens"]} tok | Gen: ~{stats["avg_gen_tokens"]} tok')
    print(f'  E2E: avg={stats["e2e_avg_ms"]:.1f}ms  p50={stats["e2e_p50_ms"]:.1f}ms  p90={stats["e2e_p90_ms"]:.1f}ms  p99={stats["e2e_p99_ms"]:.1f}ms')
    print(f'  Throughput: {stats["throughput_req_s"]} req/s = {stats["throughput_tok_s"]} tok/s')
    return stats

print('=' * 70)
print(f'Gemma 3 1B Benchmark -- Customer Workload Match')
print(f'Instance: {instance_type} | TP: {TP_DEGREE} | Batch: {BATCH_SIZE}')
print(f'Prompt: ~155 tokens | max_tokens: {MAX_TOKENS} | temp: {TEMPERATURE}')
print(f'Duration: {DURATION_SECONDS}s per level | Levels: {CONCURRENCY_LEVELS}')
print('=' * 70)

all_results = []
for conc in CONCURRENCY_LEVELS:
    print(f'\n{"=" * 50}')
    print(f'Concurrency: {conc}')
    print(f'{"=" * 50}')
    stats = await benchmark_concurrency(conc)
    if stats:
        all_results.append(stats)

print('\nBenchmark complete!')

Gemma 3 1B Benchmark -- Customer Workload Match
Instance: trn2.3xlarge | TP: 1 | Batch: 4
Prompt: ~155 tokens | max_tokens: 64 | temp: 0.0
Duration: 60s per level | Levels: [1, 2, 4, 8, 16, 32]

Concurrency: 1
  Warmup...


  Running for 60s with 1 concurrent workers...


  Sample output: "Smartphone"
  Requests: 759 | Prompt: ~163.0 tok | Gen: ~2.0 tok
  E2E: avg=79.0ms  p50=79.1ms  p90=79.6ms  p99=80.1ms
  Throughput: 12.65 req/s = 25.3 tok/s

Concurrency: 2
  Warmup...


  Running for 60s with 2 concurrent workers...


  Requests: 824 | Prompt: ~163.0 tok | Gen: ~2.0 tok
  E2E: avg=145.9ms  p50=145.9ms  p90=146.9ms  p99=147.5ms
  Throughput: 13.73 req/s = 27.5 tok/s

Concurrency: 4
  Warmup...


...

  Requests: 892 | Prompt: ~163.0 tok | Gen: ~2.0 tok
  E2E: avg=539.4ms  p50=540.3ms  p90=543.0ms  p99=544.4ms
  Throughput: 14.87 req/s = 29.7 tok/s

Concurrency: 16
  Warmup...


  Running for 60s with 16 concurrent workers...


  Requests: 900 | Prompt: ~163.0 tok | Gen: ~2.0 tok
  E2E: avg=1076.5ms  p50=1083.8ms  p90=1086.0ms  p99=1087.5ms
  Throughput: 15.0 req/s = 30.0 tok/s

Concurrency: 32
  Warmup...


  Running for 60s with 32 concurrent workers...


  Requests: 916 | Prompt: ~163.0 tok | Gen: ~2.0 tok
  E2E: avg=2133.0ms  p50=2166.2ms  p90=2173.7ms  p99=2175.2ms
  Throughput: 15.27 req/s = 30.5 tok/s

Benchmark complete!


## Step 6: Results and Cross-Platform Comparison

In [9]:
import json
from datetime import datetime

# Save results
output = {
    'benchmark': 'customer-workload-match',
    'instance_type': instance_type,
    'accelerator': 'AWS Trainium2',
    'neuron_cores': nc_count,
    'serving': 'vLLM-neuron (NxDI fork fix/gemma3-1b-oob)',
    'model': 'google/gemma-3-1b-it',
    'dtype': DTYPE,
    'quantization': 'none (BF16)',
    'tp_degree': TP_DEGREE,
    'batch_size': BATCH_SIZE,
    'max_model_len': MAX_MODEL_LEN,
    'max_num_seqs': MAX_NUM_SEQS,
    'max_tokens': MAX_TOKENS,
    'temperature': TEMPERATURE,
    'prompt_tokens': '~155',
    'speculative_decoding': False,
    'prefix_caching': False,
    'timestamp': datetime.now().isoformat(),
    'results': all_results,
}

outfile = f'/home/ubuntu/gemma3_neuron_customer_workload_{instance_type}.json'
with open(outfile, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Results saved to {outfile}')

# Summary table
print('\n' + '=' * 100)
print(f'SUMMARY: Gemma 3 1B on {instance_type} (Neuron, BF16, batch_size={BATCH_SIZE})')
print(f'Workload: ~155 input tokens -> max {MAX_TOKENS} output tokens | temp={TEMPERATURE}')
print('=' * 100)
header = f'{"Conc":>5} | {"Reqs":>6} | {"In tok":>7} | {"Out tok":>8} | {"E2E avg":>10} | {"E2E p50":>10} | {"E2E p90":>10} | {"E2E p99":>10} | {"Req/s":>7} | {"Tok/s":>7}'
print(header)
print('-' * len(header))
for r in all_results:
    print(f'{r["concurrency"]:>5} | {r["total_requests"]:>6} | '
          f'{r["avg_prompt_tokens"]:>5.0f}   | {r["avg_gen_tokens"]:>6.1f}   | '
          f'{r["e2e_avg_ms"]:>8.1f}ms | {r["e2e_p50_ms"]:>8.1f}ms | '
          f'{r["e2e_p90_ms"]:>8.1f}ms | {r["e2e_p99_ms"]:>8.1f}ms | '
          f'{r["throughput_req_s"]:>5.1f} | {r["throughput_tok_s"]:>5.1f}')

if all_results:
    peak = max(all_results, key=lambda x: x['throughput_req_s'])
    best_lat = min(all_results, key=lambda x: x['e2e_p50_ms'])
    print(f'\nBest latency: {best_lat["e2e_p50_ms"]}ms p50 @ conc={best_lat["concurrency"]}')
    print(f'Peak throughput: {peak["throughput_req_s"]} req/s @ conc={peak["concurrency"]}')

Results saved to /home/ubuntu/gemma3_neuron_customer_workload_trn2.3xlarge.json

SUMMARY: Gemma 3 1B on trn2.3xlarge (Neuron, BF16, batch_size=4)
Workload: ~155 input tokens -> max 64 output tokens | temp=0.0
 Conc |   Reqs |  In tok |  Out tok |    E2E avg |    E2E p50 |    E2E p90 |    E2E p99 |   Req/s |   Tok/s
-----------------------------------------------------------------------------------------------------------
    1 |    759 |   163   |    2.0   |     79.0ms |     79.1ms |     79.6ms |     80.1ms |  12.7 |  25.3
    2 |    824 |   163   |    2.0   |    145.9ms |    145.9ms |    146.9ms |    147.5ms |  13.7 |  27.5
    4 |    868 |   163   |    2.0   |    277.7ms |    277.8ms |    279.0ms |    279.7ms |  14.5 |  28.9
    8 |    892 |   163   |    2.0   |    539.4ms |    540.3ms |    543.0ms |    544.4ms |  14.9 |  29.7
   16 |    900 |   163   |    2.0   |   1076.5ms |   1083.8ms |   1086.0ms |   1087.5ms |  15.0 |  30.0
   32 |    916 |   163   |    2.0   |   2133.0ms |   21

In [10]:
# Cross-platform comparison
# Reference numbers from our benchmarks (April 2, 2026)

print('\n' + '=' * 95)
print('CROSS-PLATFORM COMPARISON (Customer Workload: ~155 in, ~2-3 out, max_tokens=64, temp=0)')
print('=' * 95)

baselines = {
    'Customer RTX PRO 6000 (INT4-AWQ+spec+prefix)': {'p50': 32.3, 'p99': 52.4, 'note': 'Customer production'},
    'GPU g6.xlarge L4 (BF16, vLLM, conc=1)':       {'p50': 41.1, 'p99': 42.0, 'note': 'Our GPU benchmark'},
    'GPU g6.xlarge L4 (BF16, vLLM, conc=32)':      {'p50': 206.4, 'p99': 226.8, 'note': '155 req/s'},
}

print(f'{"Platform":>55} | {"E2E p50":>10} | {"E2E p99":>10} | {"Notes":>20}')
print('-' * 105)

for name, data in baselines.items():
    print(f'{name:>55} | {data["p50"]:>8.1f}ms | {data["p99"]:>8.1f}ms | {data["note"]:>20}')

# Our results
if all_results:
    conc1 = next((r for r in all_results if r['concurrency'] == 1), all_results[0])
    peak = max(all_results, key=lambda x: x['throughput_req_s'])
    
    label1 = f'Neuron {instance_type} (BF16, vLLM-neuron, conc=1)'
    print(f'{label1:>55} | {conc1["e2e_p50_ms"]:>8.1f}ms | {conc1["e2e_p99_ms"]:>8.1f}ms | {"This notebook":>20}')
    
    label_peak = f'Neuron {instance_type} (BF16, vLLM-neuron, conc={peak["concurrency"]})'
    peak_rps_str = f'{peak["throughput_req_s"]} req/s'
    print(f'{label_peak:>55} | {peak["e2e_p50_ms"]:>8.1f}ms | {peak["e2e_p99_ms"]:>8.1f}ms | {peak_rps_str:>20}')

    # Analysis
    print('\n--- Key Findings ---')
    gpu_p50 = 41.1
    cx_p50 = 32.3
    our_p50 = conc1['e2e_p50_ms']
    gpu_peak_rps = 155.1
    our_peak_rps = peak['throughput_req_s']
    
    print(f'  Neuron vs GPU L4 (BF16): {our_p50/gpu_p50:.1f}x slower latency, {gpu_peak_rps/our_peak_rps:.0f}x lower throughput')
    print(f'  Neuron vs Customer:      {our_p50/cx_p50:.1f}x slower latency (but customer uses INT4-AWQ + speculative)')
    print(f'  Neuron p99-p50 gap:      {conc1["e2e_p99_ms"] - conc1["e2e_p50_ms"]:.1f}ms (extremely tight -- Neuron strength)')
    print(f'  GPU L4 p99-p50 gap:      {42.0 - 41.1:.1f}ms at conc=1, up to {226.8 - 206.4:.1f}ms at conc=32')
    
    print('\n--- Gaps Limiting Neuron Competitiveness ---')
    print('  1. No INT4-AWQ quantization (customer uses this for 4x smaller model)')
    print('  2. No speculative decoding (customer uses ngram 3-token lookahead)')
    print('  3. No prefix caching (customer enables this)')
    print('  4. Fixed batch compilation vs GPU dynamic batching (10x throughput gap)')
    print('  5. CTE bucket >= 512 forced by compiler issue (wastes compute on short prompts)')
    print('  6. NKI attention kernel disabled (head_dim=256 unsupported)')


CROSS-PLATFORM COMPARISON (Customer Workload: ~155 in, ~2-3 out, max_tokens=64, temp=0)
                                               Platform |    E2E p50 |    E2E p99 |                Notes
---------------------------------------------------------------------------------------------------------
           Customer RTX PRO 6000 (INT4-AWQ+spec+prefix) |     32.3ms |     52.4ms |  Customer production
                  GPU g6.xlarge L4 (BF16, vLLM, conc=1) |     41.1ms |     42.0ms |    Our GPU benchmark
                 GPU g6.xlarge L4 (BF16, vLLM, conc=32) |    206.4ms |    226.8ms |            155 req/s
        Neuron trn2.3xlarge (BF16, vLLM-neuron, conc=1) |     79.1ms |     80.1ms |        This notebook
       Neuron trn2.3xlarge (BF16, vLLM-neuron, conc=32) |   2166.2ms |   2175.2ms |          15.27 req/s

--- Key Findings ---
  Neuron vs GPU L4 (BF16): 1.9x slower latency, 10x lower throughput
  Neuron vs Customer:      2.4x slower latency (but customer uses INT4-AWQ + specula

## Cleanup

In [11]:
import signal

try:
    server_proc.send_signal(signal.SIGTERM)
    server_proc.wait(timeout=10)
    print('vLLM server stopped.')
except:
    server_proc.kill()
    print('vLLM server killed.')

print('\nDon\'t forget to terminate the instance when done!')
print(f'Instance type: {instance_type}')

vLLM server stopped.

Don't forget to terminate the instance when done!
Instance type: trn2.3xlarge


## Appendix: Known Issues and Configuration Reference

### Why These Specific Settings?

**`attn_kernel_enabled: false`**

The NKI attention kernel supports head_dim up to 128. Gemma 3 1B uses head_dim=256.
With the kernel enabled, compilation succeeds but inference produces wrong results.
Our NxDI fork auto-disables it when head_dim > 128, but we set it explicitly for clarity.

**`k_cache_transposed: true`**

The K cache transposition (`BHSD -> BHDS`) is required for correct BMM shapes with
head_dim=256. However, the base class `attention_base.py` disables transposition for SWA
layers while the KV cache manager uses the global setting. Our fork fixes this mismatch
so both SWA and global layers handle BHDS layout correctly with GQA expansion.

**`context_encoding_buckets: [512]` / `token_generation_buckets: [512]`**

The Neuron compiler generates DGE scatter/gather instructions that produce OOB memory
accesses when bucket_size < 512 AND head_dim=256. This is a compiler issue.
Using 512 as minimum bucket avoids the crash. This means all prompts (even short ones)
are padded to 512 tokens, wasting ~3x compute for the customer's ~155-token prompts.

**`batch_size: 4`**

We tested batch_size=4 vs batch_size=16 for this workload. BS16 was 27% slower at
conc=1 (100ms vs 79ms) with identical peak throughput (~15 req/s). The workload
generates only ~2 tokens, so throughput is CTE-bound, not token-generation-bound.
Larger batches add CTE padding cost without token generation benefit.

### NxDI Fork: Required Fixes

Repository: `https://github.com/jimburtoft/neuronx-distributed-inference.git`
Branch: `fix/gemma3-1b-oob`

| Commit | Fix |
|--------|-----|
| `ba0cf76` | Chunked attention for head_dim>128, vocab_size fix for 262144, auto-disable NKI, constants registration |
| `37cad61` | Fix k_cache_transposed for SWA+GQA layers (repeat_kv assumed BHSD but got BHDS) |

### Gemma 3 Variants Status on Neuron

| Variant | head_dim | Status | Issue |
|---------|----------|--------|-------|
| **1B** | 256 | Works with fork | OOB + layout fixes |
| **4B** | 256 | Broken | Same head_dim issues + missing query_pre_attn_scalar |
| **12B** | 256 | Broken | Same as 4B |
| **27B** | 128 | Broken | Missing query_pre_attn_scalar in config attributes |